# 6 · Customizing the look

Palettes, base maps, settings files, and custom colour vocabularies — the knobs for matching a brand or a report style.

In [1]:
import pathlib
import geopandas as gpd
import roadstyle as rs

# A GeoDataFrame of road edges. roadstyle only needs a *road-class* column (default "highway")
# and line geometry in any CRS — it reprojects to web coordinates (EPSG:4326) for you.
edges = gpd.read_file(pathlib.Path("data") / "sodermalm_edges.gpkg")
print(f"{len(edges):,} edges, CRS {edges.crs}")
edges[[c for c in edges.columns if c != edges.geometry.name]].head(3)


4,563 edges, CRS EPSG:4326


ERROR 1: PROJ: proj_create_from_database: Open of /home/kaveh/miniconda3/envs/roadstyle/share/proj failed


,edge_id,osm_id,edge_ref,highway,name,lanes,oneway,maxspeed_kmh,length_m,tunnel,bridge,layer
0,6379011699359957301,4080398,4080398#1f,residential,Nackagatan,1,False,30.0,61.872971,NaN,NaN,NaN
1,4910696379820034244,4080398,4080398#3f,residential,Nackagatan,1,False,30.0,34.481152,NaN,NaN,NaN
2,6981489206416339981,4080984,4080984#2f,residential,Högalidsgatan,1,False,30.0,34.356903,NaN,NaN,NaN


### Base maps
The primary base map is a *setting* (`basemap: voyager` in the bundled defaults); override it per call with any key in `rs.BASEMAPS`.

In [2]:
print("available base maps:", list(rs.BASEMAPS))
rs.render_edges(edges, basemap="positron", compress=True)

available base maps: ['voyager', 'positron', 'dark_matter', 'osm', 'esri_gray', 'satellite']


### Offer several base maps as a switcher (folium)
Pass `basemaps=[...]` to add a thumbnail base-layer switcher to the map.

In [3]:
rs.render_edges(edges, basemap="dark_matter", basemaps=["dark_matter", "positron", "satellite"], compress=True)

### Your own colour vocabulary
`color_by` + `colors` lets you define exactly what each category looks like — e.g. a congestion scheme — without registering anything.

In [4]:
import numpy as np
rng = np.random.default_rng(0)
edges["congestion"] = rng.choice(["free", "slow", "jam"], size=len(edges), p=[.6, .3, .1])
rs.render_edges(
    edges, basemap="dark_matter",
    color_by="congestion",
    colors={"free": "#22c55e", "slow": "#f59e0b", "jam": "#ef4444"},
    legend=True,
    compress=True,
)

> For *reusable* palettes shared across projects, roadstyle also has a palette registry and JSON I/O (`register_palette`, `save_palette`, `load_palette`). The data-driven `colors=` map above covers most one-off styling needs.

### Settings: one defaults file, your overrides on top

Every styling default — palettes, opacities, casing, road widths, draw order, the base map, labels, arrows, annotation slots — ships in one file, `roadstyle/data/defaults.json`. You never edit it: create a `roadstyle.json` next to your script (or in `~/.config/roadstyle/`, or point `$ROADSTYLE_CONFIG` at one) stating **only what changes**, and it merges on top at import.

From code — e.g. to restyle mid-notebook or render two looks in one process — `rs.use_settings(...)` applies the same kind of override at runtime:

In [5]:
# a settings override from code: same layout as a roadstyle.json file
rs.use_settings({
    "config": {"labels": {"color": "#8899aa"}},          # restyle street names
    "palettes": {"highsat": {"service": {"fill": "#333333"}}},
})
m = rs.render_edges(edges, basemap="dark_matter", compress=True)
rs.use_settings()      # back to the defaults
m

For a **single map**, skip the state entirely: `settings=` applies the same override for that one render and restores everything after — two calls with two `settings=` values give two looks, no cleanup needed.

In [6]:
# per-call: this render is restyled; the next render is back to the defaults
rs.render_edges(
    edges, basemap="dark_matter", compress=True,
    settings={"config": {"labels": {"color": "#8899aa"}},
              "palettes": {"highsat": {"residential": {"fill": "#9ad0ff"}}}},
)